In [9]:
import matplotlib.pyplot as plt

import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit.library import RealAmplitudes, ZZFeatureMap
# from qiskit.primitives import StatevectorSampler as Sampler
from qiskit_aer.primitives import Sampler

from qiskit_machine_learning.utils import algorithm_globals
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.connectors import TorchConnector

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from time import time

def f(x): return 1 / (1 + 25 * x**2)

N = 300
x = np.random.uniform(-1, 1, size=(N, 1))
X = np.concatenate([x, np.sqrt(1-x**2)], axis=1)
y = f(x)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_loader = DataLoader(TensorDataset(torch.Tensor(X_train), torch.Tensor(y_train)), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(torch.Tensor(X_test),  torch.Tensor(y_test)),  batch_size=64)

In [10]:
num_inputs = 2

# Define feature map and ansatz
feature_map = ZZFeatureMap(num_inputs)
ansatz = RealAmplitudes(num_inputs, entanglement="linear", reps=1)

# Define quantum circuit of num_qubits = input dim
# Append feature map and ansatz
qc = QuantumCircuit(num_inputs)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)

sampler = Sampler()

qnn = SamplerQNN(
    circuit=qc,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
    sampler=sampler,
    output_shape=4,
    input_gradients=True
)

# Set up PyTorch module
# Reminder: If we don't explicitly declare the initial weights
# they are chosen uniformly at random from [-1, 1].
initial_weights = 0.1 * (2 * algorithm_globals.random.random(qnn.num_weights) - 1)
print("Initial weights: ", initial_weights)
qnn_torch = TorchConnector(qnn, initial_weights)

C:\Users\purav\AppData\Local\Temp\ipykernel_20816\2155182732.py:15: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  qnn = SamplerQNN(
No interpret function given, output_shape will be automatically determined as 2^num_virtual_qubits.


Initial weights:  [ 0.04342051 -0.00553105  0.06313981 -0.07551009]


In [11]:
class QuantumApprox(nn.Module):
    def __init__(self):
        super().__init__()
        self.qnn = qnn_torch
        self.fc_out = nn.Linear(4, 1)

    def forward(self, x):
        x = self.qnn(x)
        x = torch.tanh(x)
        x = self.fc_out(x)
        return x

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model = QuantumApprox().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.MSELoss()


def train():
    model.train()
    
    numerator = 0.0
    denominator = 0.0
    
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(Xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        # Detach and move to CPU for numpy operations
        pred_np = pred.detach().cpu().numpy()
        yb_np = yb.detach().cpu().numpy()

        numerator += np.linalg.norm(pred_np - yb_np) ** 2
        denominator += np.linalg.norm(yb_np) ** 2

    return  np.sqrt(numerator) / np.sqrt(denominator)   # L2 Relative Error

def evaluate(loader):
    model.eval()
    
    numerator = 0.0
    denominator = 0.0
    
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            pred = model(Xb)
        
            # Detach and move to CPU for numpy operations
            pred_np = pred.detach().cpu().numpy()
            yb_np = yb.detach().cpu().numpy()

            numerator += np.linalg.norm(pred_np - yb_np) ** 2
            denominator += np.linalg.norm(yb_np) ** 2
            
    return np.sqrt(numerator) / np.sqrt(denominator)   # L2 Relative Error

for epoch in range(1, 3):
    start = time()
    train_rmse = train()
    test_rmse = evaluate(test_loader)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:2d}: Train L2 Relative={train_rmse:.4f}; Test L2 Relative={test_rmse:.4f}")
    print(time() - start)
    start = 0


cpu
11.416934490203857
11.277930736541748


In [ ]:
"""
testing = np.linspace(-1, 1, 75)
answer = [f(x) for x in testing]
plt.plot(testing, answer)

testing = testing.reshape(-1, 1)
xd = np.concatenate([testing, testing ** 2, testing ** 3], axis=1)
print(testing.shape)
out = model(torch.from_numpy(xd).float()).cpu().detach().numpy()
plt.plot(testing, out)
"""